# PRLC ResNet50 CIFAR-10 Training Script 
### EquiAdapt + ImageNet ResNet50 + C8 canonicalization

In [ ]:
!pip install equiadapt

In [ ]:
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from torchvision.models import resnet50, ResNet50_Weights

from omegaconf import OmegaConf

from equiadapt.images.canonicalization.discrete_group import (
    GroupEquivariantImageCanonicalization,
)
from equiadapt.images.canonicalization_networks import ESCNNEquivariantNetwork


# ============================================================
# Config
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-4

NUM_CLASSES = 10
IMAGE_SHAPE = (3, 224, 224)

GROUP_TYPE = "rotation"
NUM_ROTATIONS = 8
BETA = 100.0

PRINT_EVERY = 100

SAVE_PATH = "/kaggle/working/prlc_resnet50_cifar10_c8.pt"


# ============================================================
# PRLC wrapper
# ============================================================

class PRLCGroupEquivariantImageCanonicalization(GroupEquivariantImageCanonicalization):
    def get_prior_regularization_loss(self):
        """
        Discrete C8 prior from the paper:
        L_prior = -log p(identity rotation)

        Identity rotation is index 0.
        """
        if not hasattr(self, "canonicalization_info_dict"):
            raise RuntimeError(
                "Call canonicalizer(inputs) before get_prior_regularization_loss()."
            )

        logits = self.canonicalization_info_dict["group_activations"]

        identity_targets = torch.zeros(
            logits.shape[0],
            dtype=torch.long,
            device=logits.device,
        )

        return F.cross_entropy(logits, identity_targets)

    def add_prior_regularizer(self, loss):
        return loss + self.beta * self.get_prior_regularization_loss()


# ============================================================
# Helpers
# ============================================================

def format_time(seconds):
    seconds = int(seconds)
    minutes = seconds // 60
    seconds = seconds % 60

    if minutes >= 60:
        hours = minutes // 60
        minutes = minutes % 60
        return f"{hours}h {minutes}m {seconds}s"

    return f"{minutes}m {seconds}s"


@torch.no_grad()
def evaluate(loader):
    prediction_network.eval()
    canonicalizer.eval()

    correct = 0
    total = 0

    for inputs, targets in loader:
        inputs = inputs.to(device)
        targets = targets.to(device)

        inputs_canonicalized = canonicalizer(inputs)
        outputs = prediction_network(inputs_canonicalized)

        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

    return correct / total


# ============================================================
# Data
# ============================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(
        degrees=10,
        interpolation=InterpolationMode.BILINEAR,
        fill=0,
    ),
    transforms.Resize(232, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

test_transform = transforms.Compose([
    transforms.Resize(232, interpolation=InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

train_dataset = datasets.CIFAR10(
    root="/kaggle/working/data",
    train=True,
    download=True,
    transform=train_transform,
)

test_dataset = datasets.CIFAR10(
    root="/kaggle/working/data",
    train=False,
    download=True,
    transform=test_transform,
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)


# ============================================================
# Canonicalizer: C8 equivariant network
# ============================================================

canonicalization_network = ESCNNEquivariantNetwork(
    IMAGE_SHAPE,
    16,
    3,
    GROUP_TYPE,
    NUM_ROTATIONS,
    3,
).to(device)

canonicalization_hyperparams = OmegaConf.create({
    "group_type": GROUP_TYPE,
    "num_rotations": NUM_ROTATIONS,
    "alpha": 1.0,
    "beta": BETA,
    "input_crop_ratio": 0.9,
    "resize_shape": [224, 224],
})

canonicalizer = PRLCGroupEquivariantImageCanonicalization(
    canonicalization_network=canonicalization_network,
    canonicalization_hyperparams=canonicalization_hyperparams,
    in_shape=IMAGE_SHAPE,
).to(device)


# ============================================================
# Prediction network: ImageNet-pretrained ResNet50
# ============================================================

prediction_network = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)

prediction_network.fc = nn.Linear(
    prediction_network.fc.in_features,
    NUM_CLASSES,
)

prediction_network = prediction_network.to(device)


# ============================================================
# Optimizer
# ============================================================

optimizer = optim.Adam(
    list(prediction_network.parameters()) + list(canonicalizer.parameters()),
    lr=LR,
)


# ============================================================
# Initial logs
# ============================================================

print("====================================")
print("PRLC ResNet50 CIFAR-10 training")
print("====================================")
print(f"Device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")
print(f"Train batches per epoch: {len(train_loader)}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LR}")
print(f"Group: C{NUM_ROTATIONS}")
print(f"Beta: {BETA}")
print(f"Save path: {SAVE_PATH}")
print("====================================")


# ============================================================
# Training
# ============================================================

training_start_time = time.time()

for epoch in range(EPOCHS):
    epoch_start_time = time.time()

    prediction_network.train()
    canonicalizer.train()

    total_loss_sum = 0.0
    task_loss_sum = 0.0
    prior_term_sum = 0.0

    correct = 0
    total = 0

    print(f"\nStarting epoch {epoch + 1}/{EPOCHS}")

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        batch_start_time = time.time()

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        inputs_canonicalized = canonicalizer(inputs)
        outputs = prediction_network(inputs_canonicalized)

        task_loss = F.cross_entropy(outputs, targets)

        loss = canonicalizer.add_prior_regularizer(task_loss)

        loss.backward()
        optimizer.step()

        prior_term = loss.detach() - task_loss.detach()

        total_loss_sum += loss.item()
        task_loss_sum += task_loss.item()
        prior_term_sum += prior_term.item()

        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        if batch_idx == 0 or (batch_idx + 1) % PRINT_EVERY == 0:
            batches_done = batch_idx + 1

            running_loss = total_loss_sum / batches_done
            running_task_loss = task_loss_sum / batches_done
            running_prior_term = prior_term_sum / batches_done
            running_acc = correct / total

            elapsed_epoch = time.time() - epoch_start_time
            avg_batch_time = elapsed_epoch / batches_done
            remaining_batches = len(train_loader) - batches_done
            eta = remaining_batches * avg_batch_time

            batch_time = time.time() - batch_start_time

            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Batch {batches_done}/{len(train_loader)} | "
                f"Loss: {running_loss:.4f} | "
                f"Task: {running_task_loss:.4f} | "
                f"PriorTerm: {running_prior_term:.4f} | "
                f"Train Acc: {running_acc:.4f} | "
                f"Batch Time: {batch_time:.2f}s | "
                f"ETA: {format_time(eta)}",
                flush=True,
            )

    train_acc = correct / total
    train_loss = total_loss_sum / len(train_loader)
    train_task_loss = task_loss_sum / len(train_loader)
    train_prior_term = prior_term_sum / len(train_loader)

    print("Evaluating on test set...")
    test_acc = evaluate(test_loader)

    epoch_time = time.time() - epoch_start_time
    total_time = time.time() - training_start_time

    print("------------------------------------")
    print(f"Epoch {epoch + 1}/{EPOCHS} complete")
    print(f"Train Acc:   {train_acc:.4f}")
    print(f"Test Acc:    {test_acc:.4f}")
    print(f"Loss:        {train_loss:.4f}")
    print(f"Task Loss:   {train_task_loss:.4f}")
    print(f"Prior Term:  {train_prior_term:.4f}")
    print(f"Epoch Time:  {format_time(epoch_time)}")
    print(f"Total Time:  {format_time(total_time)}")
    print("------------------------------------")


# ============================================================
# Save final model
# ============================================================

torch.save(
    {
        "prediction_network_state_dict": prediction_network.state_dict(),
        "canonicalizer_state_dict": canonicalizer.state_dict(),
        "canonicalization_network_state_dict": canonicalization_network.state_dict(),
        "config": {
            "dataset": "CIFAR10",
            "model": "PRLC ResNet50",
            "group_type": GROUP_TYPE,
            "num_rotations": NUM_ROTATIONS,
            "beta": BETA,
            "epochs": EPOCHS,
            "lr": LR,
            "batch_size": BATCH_SIZE,
        },
    },
    SAVE_PATH,
)

print(f"\nSaved model to {SAVE_PATH}")